Goal: Build two data prep pipelines using different datasets to get practice with data preparation and question building.


**Step one**: Review these two datasets and brainstorm problems that could be addressed with the dataset. Identify a question for each dataset. 

[College Completion Data Dictionary + Data](https://www.kaggle.com/datasets/thedevastator/boost-student-success-with-college-completion-da/data)

  - [Dataset is located here](https://github.com/UVADS/DS-3021/blob/main/data/cc_institution_details.csv)

[Job_Placement](https://raw.githubusercontent.com/DG1606/CMS-R-2020/master/Placement_Data_Full_Class.csv) 

 - [Data Dictionary (kinda) for Job Placement](https://www.kaggle.com/benroshan/factors-affecting-campus-placement/discussion/280612) - You'll need to infer from the column names but also the comments on the site.

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# Dataset 1 - College details - What are the statistical differences between private and public colleges?
college = pd.read_csv('cc_institution_details.csv')

# Dataset 2 - Job placement - Which majors produce the most money?
placement = pd.read_csv('Placement_Data_Full_Class.csv') 
print(placement.shape)
placement.head()

(215, 15)


,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status,salary
0,1,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed,270000.0
1,2,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed,200000.0
2,3,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed,250000.0
3,4,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed,NaN
4,5,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed,425000.0


Dataset 1 - College Data

Step two: Work through the steps outlined in the examples to include the following elements: 

  * Write a generic question that this dataset could address. - Do private colleges have higher graduation rates than public colleges?
  * What is a independent Business Metric for your problem? Think about the case study examples we have discussed in class. - Transfer rate by institution type
  * Data preparation - Done below :
    * correct variable type/class as needed
    * collapse factor levels as needed
    * one-hot encoding factor variables 
    * normalize the continuous variables
    * drop unneeded variables
    * create target variable if needed
    * Calculate the prevalence of the target variable 
    * Create the necessary data partitions (Train,Tune,Test)

In [4]:
# correct variable type/class as needed
for each in college.columns:
    if college[each].dtype == 'object':
        college[each] = college[each].astype('category')
    if college[each].dtype == 'int64':
        college[each] = college[each].astype('float64')
# collapse factor levels as needed
college = college[college['level'] == '4-year']
# one-hot encoding factor variables
college["private_binary"] = college["control"].map({
    "Public": 0,
    "Private not-for-profit": 1,
    "Private for-profit": 1
})
# normalize the continuous variables
college_numeric = college.select_dtypes(include=['float64', 'int64'])
college[college_numeric.columns] = (college_numeric - college_numeric.mean()) / college_numeric.std()
# drop unneeded variables
college = college.drop(columns = ['chronname', 'city', 'state', 'hbcu', 'long_x', 'lat_y', 'site','vsa_year', 'vsa_grad_after4_first', 'vsa_grad_elsewhere_after4_first', 'vsa_enroll_after4_first', 'vsa_enroll_elsewhere_after4_first', 'vsa_grad_after6_first', 'vsa_grad_elsewhere_after6_first', 'vsa_enroll_after6_first', 'vsa_enroll_elsewhere_after6_first', 'vsa_grad_after4_transfer', 'vsa_grad_elsewhere_after4_transfer', 'vsa_enroll_after4_transfer', 'vsa_enroll_elsewhere_after4_transfer', 'vsa_grad_after6_transfer', 'vsa_grad_elsewhere_after6_transfer', 'vsa_enroll_after6_transfer', 'vsa_enroll_elsewhere_after6_transfer'], errors='ignore')
print(college.columns)
# create target variable if needed
college["target"] = college["grad_100_value"]
# calculate the prevalence of the target variable
prevalence = college.groupby("private_binary")["target"].mean()
print(f"Prevalence of target variable: {prevalence}")
# create the necessary data partitions (Train,Tune,Test)
train, temp = train_test_split( college, test_size=0.4, random_state=42 )
tune, test = train_test_split( temp, test_size=0.5, random_state=42 )
print(f"Train size: {train.shape}, Tune size: {tune.shape}, Test size: {test.shape}")

Index(['index', 'unitid', 'level', 'control', 'basic', 'flagship',
       'student_count', 'awards_per_value', 'awards_per_state_value',
       'awards_per_natl_value', 'exp_award_value', 'exp_award_state_value',
       'exp_award_natl_value', 'exp_award_percentile', 'ft_pct', 'fte_value',
       'fte_percentile', 'med_sat_value', 'med_sat_percentile', 'aid_value',
       'aid_percentile', 'endow_value', 'endow_percentile', 'grad_100_value',
       'grad_100_percentile', 'grad_150_value', 'grad_150_percentile',
       'pell_value', 'pell_percentile', 'retain_value', 'retain_percentile',
       'ft_fac_value', 'ft_fac_percentile', 'similar', 'state_sector_ct',
       'carnegie_ct', 'counted_pct', 'nicknames', 'cohort_size',
       'private_binary'],
      dtype='str')
Prevalence of target variable: private_binary
-1.643105   -0.329565
 0.608344    0.131231
Name: target, dtype: float64
Train size: (1403, 41), Tune size: (468, 41), Test size: (468, 41)


Dataset 2 - Job Placement

Step two: Work through the steps outlined in the examples to include the following elements: 

  * Write a generic question that this dataset could address? How does the students degree effect their salary?
  * What is a independent Business Metric for your problem? Think about the case study examples we have discussed in class. - Average starting salary of graduates by degree
  * Data preparation - Done Below
    * correct variable type/class as needed
    * collapse factor levels as needed
    * one-hot encoding factor variables 
    * normalize the continuous variables
    * drop unneeded variables
    * create target variable if needed
    * Calculate the prevalence of the target variable 
    * Create the necessary data partitions (Train,Tune,Test)

In [5]:
print(placement.columns)
# correct variable type/class as needed
for each in placement.columns:
    if placement[each].dtype == 'object':
        placement[each] = placement[each].astype('category')
    if placement[each].dtype == 'int64':
        placement[each] = placement[each].astype('float64')
# collapse factor levels as needed
placement = placement[placement["status"] == "Placed"]
# one-hot encoding factor variables
placement["status"] = 1
# normalize the continuous variables - no continuous variables after dropping columns
#placement_numeric = placement.select_dtypes(include=['float64', 'int64'])
#placement[placement_numeric.columns] = (placement_numeric - placement_numeric.mean()) / placement_numeric.std()
# drop unneeded variables - the only column with missing values that are needing to be dropped is salary however this is because the students are not placed into a job yet so I am still going to keep the column.
#placement = placement.drop(columns = ['sl_no', 'gender', 'ssc_p', 'ssc_b', 'hsc_p', 'hsc_b', 'hsc_s', 'workex', 'etest_p', 'mba_p',], errors='ignore')
# create target variable if needed
placement['target'] = placement['salary']
# calculate the prevalence of the target variable 
prevalence = placement.groupby('degree_t')['salary'].mean()
print(f"Prevalence of target variable: {prevalence}")
# create the necessary data partitions (Train,Tune,Test)
print(placement.shape)
placement = pd.get_dummies(placement, columns=["degree_t"], drop_first=True)
#train, temp = train_test_split( placement, test_size=0.4, random_state=42 ) 
#tune, test = train_test_split( temp, test_size=0.5, random_state=42 ) 
#print(train.shape, tune.shape, test.shape)


Index(['sl_no', 'gender', 'ssc_p', 'ssc_b', 'hsc_p', 'hsc_b', 'hsc_s',
       'degree_p', 'degree_t', 'workex', 'etest_p', 'specialisation', 'mba_p',
       'status', 'salary'],
      dtype='str')
Prevalence of target variable: degree_t
Comm&Mgmt    278627.450980
Others       280400.000000
Sci&Tech     314609.756098
Name: salary, dtype: float64
(148, 16)


Step three: What do your instincts tell you about the data. Can it address your problem, what areas/items are you worried about?

Step four: Create functions for your two pipelines that produces the train and test datasets. The end result should be a series of functions that can be called to produce the train and test datasets for each of your two problems that includes all the data prep steps you took. This is essentially creating a DAG for your data prep steps. Imagine you will need to do this for multiple problems in the future so creating functions that can be reused is important. You don't need to create one full pipeline function that does everything but rather a series of smaller functions that can be called in sequence to produce the final datasets. Use your judgement on how to break up the functions. 



In [6]:
# correct variable type/class as needed
def correct_variable_types(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].astype('category')
        elif df[col].dtype == 'int64':
            df[col] = df[col].astype('float64')
    return df

# collapse factor levels as needed
def collapse_factor_levels(df, column, mapping):
    df[column] = df[column].map(mapping)
    return df
# one-hot encoding factor variables 
def one_hot_encode(df, column, mapping):
    dummies = pd.get_dummies(df[column], prefix=column, drop_first=True) 
    df = pd.concat([df.drop(columns=[column]), dummies], axis=1) 
    return df
# normalize the continuous variables
def normalize_continuous_variables(df):
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
    df[numeric_cols] = (df[numeric_cols] - df[numeric_cols].mean()) / df[numeric_cols].std()
    return df
# drop unneeded variables
def drop_unneeded_variables(df, columns):
    df = df.drop(columns=columns, errors='ignore')
    return df
# create target variable if needed
def create_target_variable(df, target_column):
    df['target'] = df[target_column]
    return df
# Calculate the prevalence of the target variable 
def calculate_prevalence(df, target_column):
    prevalence = df.groupby(target_column).size()
    return prevalence
# Create the necessary data partitions (Train,Tune,Test)
def create_data_partitions(df, test_size=0.4, random_state=42):
    train, temp = train_test_split(df, test_size=test_size, random_state=random_state)
    tune, test = train_test_split(temp, test_size=0.5, random_state=random_state)
    return train, tune, test